[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/littlekg87/lecture/blob/main/2025_kmooc/notebooks/02_gu_convert.ipynb)


# 텍스트 마이닝 ② — 한문 숫자를 아라비아 숫자로 바꾸기

앞 실습에서 뽑아낸 `一十萬七百九十` 같은 한문 숫자를 `100790` 처럼 계산할 수 있는 숫자로 바꿉니다.

**왼쪽의 ▶ 버튼을 위에서부터 차례대로 누르기만 하면 됩니다.**

> 설치할 것이 없습니다. 파이썬 기본 기능만 사용합니다.


## 0단계 — 실습 데이터 내려받기

① 실습에서 만든 `sejong_gu_data_sample.csv` 를 내려받습니다.
(① 실습을 방금 마쳤다면 그때 만든 파일을 올려서 써도 됩니다.)


In [ ]:
!wget -q -O sejong_gu_data_sample.csv "https://raw.githubusercontent.com/littlekg87/lecture/main/2025_kmooc/data/text-mining/sejong_gu_data_sample.csv"

print('내려받기 완료!')

# (선택) ① 실습에서 직접 만든 파일을 올리려면 아래 두 줄의 # 을 지우고 실행하세요.
# from google.colab import files
# files.upload()


## 1단계 — 한문 숫자 매핑표 만들기

| 종류 | 글자 |
|---|---|
| 낱자 | 零〇一二三四五六七八九 |
| 작은 단위 | 十(10) 百(100) 千(1000) |
| 큰 단위 | 萬(10⁴) 億(10⁸) 兆(10¹²) |


In [ ]:
digits = {
    '零': 0, '〇': 0, '一': 1, '二': 2, '三': 3, '四': 4,
    '五': 5, '六': 6, '七': 7, '八': 8, '九': 9
}

units = {
    '十': 10,
    '百': 100,
    '千': 1000
}

large_units = {
    '兆': 1000000000000,
    '億': 100000000,
    '萬': 10000
}

print('매핑표 준비 완료!')


## 2단계 — 변환 함수 만들기

- `parse_section` : 萬 이하의 작은 덩어리를 숫자로 바꿉니다. (예: 七百九十 → 790)
- `hanja_to_number` : 萬·億·兆 를 기준으로 문자열을 잘라 각 덩어리를 더합니다.


In [ ]:
import re

def parse_section(section):
    """한문 숫자 섹션을 아라비아 숫자로 변환"""
    result = 0
    num = 0
    for char in section:
        if char in digits:
            num = digits[char]
        elif char in units:
            unit = units[char]
            if num == 0:
                num = 1
            result += num * unit
            num = 0
        else:
            pass  # 알 수 없는 문자 무시
    result += num
    return result


def hanja_to_number(hanja_str):
    """전체 한문 숫자 문자열을 아라비아 숫자로 변환"""
    hanja_str = re.sub(r'[^零〇一二三四五六七八九十百千萬億兆]', '', hanja_str)
    total = 0
    parts = re.split('([兆億萬])', hanja_str)
    parts.append('')  # 마지막 단위 처리용
    i = 0
    while i < len(parts):
        section = parts[i]
        if i + 1 < len(parts) and parts[i + 1] in large_units:
            unit = large_units[parts[i + 1]]
            total += parse_section(section) * unit
            i += 2
        else:
            total += parse_section(section)
            i += 1
    return total


# 잘 되는지 바로 확인해 봅시다
print(hanja_to_number('一十萬七百九十'))   # 100790
print(hanja_to_number('二萬四千一百七十'))  # 24170


## 3단계 — CSV 파일 불러오기


In [ ]:
import csv

input_csv = 'sejong_gu_data_sample.csv'

gu_data = []
with open(input_csv, mode='r', encoding='utf-8-sig') as file:
    reader = csv.reader(file)
    header = next(reader)  # 헤더 저장
    for row in reader:
        gu_data.append(row)

print(f'{len(gu_data)}개의 데이터를 불러왔습니다.')
print(gu_data[:5])


## 4단계 — 한문 숫자 변환


In [ ]:
converted_data = []
for row in gu_data:
    han_number = row[0]  # 1열(첫 번째 열)에 한문 숫자가 있다고 가정
    arabic_number = hanja_to_number(han_number)
    converted_data.append([han_number, arabic_number])
    print(f'  {han_number}  ->  {arabic_number}')


## 5단계 — 변환된 데이터 다시 CSV로 저장


In [ ]:
output_csv = 'sejong_gu_data_converted.csv'

with open(output_csv, mode='w', newline='', encoding='utf-8-sig') as file:
    writer = csv.writer(file)
    writer.writerow(['구(口) 데이터 (한문)', '구(口) 데이터 (아라비아 숫자)'])
    for row in converted_data:
        writer.writerow(row)

print(f'한문 숫자가 변환되어 {output_csv} 파일로 저장되었습니다!')


## 6단계 — 만든 파일 내 컴퓨터로 내려받기


In [ ]:
from google.colab import files

files.download('sejong_gu_data_converted.csv')


---
다음 실습 → **[단어 분석 ① 단어 빈도 분석](03_word_frequency.ipynb)**
